In [1]:
import re

import polars as pl
from rdkit import Chem
from rdkit.Chem.Descriptors import MolWt
from rdkit.Chem.rdMolDescriptors import CalcMolFormula

In [2]:
def formula_from_smiles(smiles: str) -> str | None:
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return CalcMolFormula(mol)

def get_atom_count_from_formula(formula: str, atom: str) -> int:
    if formula is None:
        return 0
    match = re.search(rf"{atom}(?![a-z])(\d*)", formula)
    if not match:
        return 0
    n = match.group(1)
    return int(n) if n else 1

In [3]:
df = pl.read_parquet("../data/processed/deduplicated.parquet")

In [4]:
df.head(10)

name,smiles,mpC
str,str,f64
"""cyclobutylmethane""","""C1(CCC1)C""",-161.51
"""Nitrogen oxide""","""[O-][N+]#N""",-90.8
"""Sulfuryl difluoride""","""FS(F)(=O)=O""",-135.8
"""disopyramide""","""CC(C)N(CCC(c1ccccn1)(c2ccccc2)…",94.8
"""Bromine""","""BrBr""",-7.2
"""Lomefloxacin""","""O=C(O)C2=CN(CC)c1c(F)c(c(F)cc1…",239.75
"""N,N-Dimethylmethanamine""","""CN(C)C""",-117.0
"""Tetrachloromethane""","""ClC(Cl)(Cl)Cl""",-23.0
"""Iodine""","""II""",113.5


In [5]:
df = df.with_columns(
    pl.col("smiles").map_elements(formula_from_smiles, return_dtype=pl.String).alias("formula")
)

[12:37:07] Can't kekulize mol.  Unkekulized atoms: 0 1 2 3 4
[12:37:07] Can't kekulize mol.  Unkekulized atoms: 2 3 4 5 6
[12:37:07] Can't kekulize mol.  Unkekulized atoms: 24 25 26 27 28 31 32 33 34
[12:37:07] Can't kekulize mol.  Unkekulized atoms: 0 1 2 3 4
[12:37:07] Can't kekulize mol.  Unkekulized atoms: 0 1 2 3 4 5 6 7 8
[12:37:07] Can't kekulize mol.  Unkekulized atoms: 1 2 3 4 5 6 7 8 9
[12:37:07] Can't kekulize mol.  Unkekulized atoms: 16 17 18 19 20 21 22 23 24
[12:37:07] Can't kekulize mol.  Unkekulized atoms: 3 4 5 6 7 8 9 10 11
[12:37:07] Can't kekulize mol.  Unkekulized atoms: 3 4 5 6 8
[12:37:07] Can't kekulize mol.  Unkekulized atoms: 0 1 2 3 4 5 6 7 8
[12:37:07] Can't kekulize mol.  Unkekulized atoms: 1 2 3 4 5 6 7 8 9
[12:37:07] Can't kekulize mol.  Unkekulized atoms: 0 1 2 3 4 5 6 7 8
[12:37:07] Can't kekulize mol.  Unkekulized atoms: 0 1 2 3 12 13 14 15 16
[12:37:07] Can't kekulize mol.  Unkekulized atoms: 0 1 2 3 4 5 6 7 8
[12:37:07] Can't kekulize mol.  Unkekuliz

In [6]:
df.height

3040

In [7]:
df.filter(pl.col("formula").is_null())

name,smiles,mpC,formula
str,str,f64,str
"""1H-Pyrrole""","""c1cccn1""",-23.0,null
"""fludioxonil""","""N#Cc3cncc3c1cccc2OC(F)(F)Oc12""",199.4,null
"""Methyl (3beta,16beta,17alpha,1…","""COc1cc(cc(OC)c1OC)/C=C/C(=O)O[…",238.0,null
"""1H-Imidazole""","""c1cncn1""",90.0,null
"""1H-Indole""","""c1cccc2nccc12""",52.0,null
…,…,…,…
"""1H-Benzotriazole""","""c1cccc2nnnc12""",97.0,null
"""N-Benzyl-9H-purin-6-amine""","""n2cnc(NCc1ccccc1)c3ncnc23""",231.0,null
"""2-(1,3-Thiazol-4-yl)-1H-benzim…","""n1c3ccccc3nc1c2cscn2""",304.5,null


In [8]:
df = df.drop_nulls(subset=["formula"])
df.height

3024

In [9]:
atoms = ["C", "H", "O", "N", "S", "F", "Cl", "Br", "I"]

for atom in atoms:
    df = df.with_columns(
        pl.col("formula")
        .map_elements(lambda f: get_atom_count_from_formula(f, atom), return_dtype=pl.Int32)
        .alias(f"{atom}_count")
    )

df.head(10)

name,smiles,mpC,formula,C_count,H_count,O_count,N_count,S_count,F_count,Cl_count,Br_count,I_count
str,str,f64,str,i32,i32,i32,i32,i32,i32,i32,i32,i32
"""cyclobutylmethane""","""C1(CCC1)C""",-161.51,"""C5H10""",5,10,0,0,0,0,0,0,0
"""Nitrogen oxide""","""[O-][N+]#N""",-90.8,"""N2O""",0,0,1,2,0,0,0,0,0
"""Sulfuryl difluoride""","""FS(F)(=O)=O""",-135.8,"""F2O2S""",0,0,2,0,1,2,0,0,0
"""disopyramide""","""CC(C)N(CCC(c1ccccn1)(c2ccccc2)…",94.8,"""C21H29N3O""",21,29,1,3,0,0,0,0,0
"""Bromine""","""BrBr""",-7.2,"""Br2""",0,0,0,0,0,0,0,2,0
"""Lomefloxacin""","""O=C(O)C2=CN(CC)c1c(F)c(c(F)cc1…",239.75,"""C17H19F2N3O3""",17,19,3,3,0,2,0,0,0
"""N,N-Dimethylmethanamine""","""CN(C)C""",-117.0,"""C3H9N""",3,9,0,1,0,0,0,0,0
"""Tetrachloromethane""","""ClC(Cl)(Cl)Cl""",-23.0,"""CCl4""",1,0,0,0,0,0,4,0,0
"""Iodine""","""II""",113.5,"""I2""",0,0,0,0,0,0,0,0,2


In [10]:
df = df.with_columns(
    pl.col("smiles")
    .map_elements(
        lambda s: MolWt(mol) if (mol := Chem.MolFromSmiles(s)) is not None else None,
        return_dtype=pl.Float64
    )
    .alias("molecular_weight")
)

In [11]:
df = df.with_columns(
    pl.col("smiles")
    .map_elements(
        lambda s: sum(1 for atom in mol.GetAtoms() if atom.GetDegree() > 2 and not atom.IsInRing())
        if (mol := Chem.MolFromSmiles(s)) is not None else None,
        return_dtype=pl.Int32
    )
    .alias("branch_count")
)

In [12]:
df = df.with_columns(
    pl.col("smiles")
    .map_elements(
        lambda s: len(Chem.GetSSSR(mol)) if (mol := Chem.MolFromSmiles(s)) is not None else None,
        return_dtype=pl.Int32
    )
    .alias("cycle_count")
)

In [13]:
df = df.with_columns(
    pl.col("smiles")
    .map_elements(
        lambda s: Chem.rdMolDescriptors.CalcNumAromaticRings(mol) if (mol := Chem.MolFromSmiles(s)) is not None else None,
        return_dtype=pl.Int32
    )
    .alias("aromatic_ring_count")
)

In [14]:
df = df.with_columns(
    pl.col("smiles")
    .map_elements(
        lambda s: sum(1 for bond in mol.GetBonds() if bond.GetBondTypeAsDouble() == 2.0)
        if (mol := Chem.MolFromSmiles(s)) is not None else None,
        return_dtype=pl.Int32
    )
    .alias("double_bond_count")
)

In [15]:
df = df.with_columns(
    pl.col("smiles")
    .map_elements(
        lambda s: sum(1 for bond in mol.GetBonds() if bond.GetBondTypeAsDouble() == 3.0)
        if (mol := Chem.MolFromSmiles(s)) is not None else None,
        return_dtype=pl.Int32
    )
    .alias("triple_bond_count")
)

In [16]:
df = df.with_columns(
    pl.col("smiles")
    .map_elements(
        lambda s: len(mol.GetSubstructMatches(Chem.MolFromSmarts("C(=O)O")))
        if (mol := Chem.MolFromSmiles(s)) is not None else None,
        return_dtype=pl.Int32
    )
    .alias("carboxylic_acid_count")
)

In [17]:
df = df.with_columns(
    pl.col("smiles")
    .map_elements(
        lambda s: len(mol.GetSubstructMatches(Chem.MolFromSmarts("[OX2H]")))
        if (mol := Chem.MolFromSmiles(s)) is not None else None,
        return_dtype=pl.Int32
    )
    .alias("alcohol_count")
)

In [18]:
df = df.with_columns(
    pl.col("smiles")
    .map_elements(
        lambda s: len(mol.GetSubstructMatches(Chem.MolFromSmarts("[CX3]=[OX1]")))
        if (mol := Chem.MolFromSmiles(s)) is not None else None,
        return_dtype=pl.Int32
    )
    .alias("carbonyl_count")
)

In [19]:
df = df.with_columns(
    pl.col("smiles")
    .map_elements(
        lambda s: len(mol.GetSubstructMatches(Chem.MolFromSmarts("[NX3;H2,H1;!$([N]~[!#6]);!$([N]*~[#7,#8,#15,#16])]")))
        if (mol := Chem.MolFromSmiles(s)) is not None else None,
        return_dtype=pl.Int32
    )
    .alias("prim_sec_amine_count")
)

In [20]:
df = df.with_columns(
    pl.col("smiles")
    .map_elements(
        lambda s: len(mol.GetSubstructMatches(Chem.MolFromSmarts("[NX3][CX3](=[OX1])")))
        if (mol := Chem.MolFromSmiles(s)) is not None else None,
        return_dtype=pl.Int32
    )
    .alias("amide_count")
)

In [21]:
df.head(10)

name,smiles,mpC,formula,C_count,H_count,O_count,N_count,S_count,F_count,Cl_count,Br_count,I_count,molecular_weight,branch_count,cycle_count,aromatic_ring_count,double_bond_count,triple_bond_count,carboxylic_acid_count,alcohol_count,carbonyl_count,prim_sec_amine_count,amide_count
str,str,f64,str,i32,i32,i32,i32,i32,i32,i32,i32,i32,f64,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32
"""cyclobutylmethane""","""C1(CCC1)C""",-161.51,"""C5H10""",5,10,0,0,0,0,0,0,0,70.135,0,1,0,0,0,0,0,0,0,0
"""Nitrogen oxide""","""[O-][N+]#N""",-90.8,"""N2O""",0,0,1,2,0,0,0,0,0,44.013,0,0,0,0,1,0,0,0,0,0
"""Sulfuryl difluoride""","""FS(F)(=O)=O""",-135.8,"""F2O2S""",0,0,2,0,1,2,0,0,0,102.061,1,0,0,2,0,0,0,0,0,0
"""disopyramide""","""CC(C)N(CCC(c1ccccn1)(c2ccccc2)…",94.8,"""C21H29N3O""",21,29,1,3,0,0,0,0,0,339.483,5,2,2,1,0,0,0,1,0,1
"""Bromine""","""BrBr""",-7.2,"""Br2""",0,0,0,0,0,0,0,2,0,159.808,0,0,0,0,0,0,0,0,0,0
"""Lomefloxacin""","""O=C(O)C2=CN(CC)c1c(F)c(c(F)cc1…",239.75,"""C17H19F2N3O3""",17,19,3,3,0,2,0,0,0,351.353,1,3,2,2,0,1,1,1,1,0
"""N,N-Dimethylmethanamine""","""CN(C)C""",-117.0,"""C3H9N""",3,9,0,1,0,0,0,0,0,59.112,1,0,0,0,0,0,0,0,0,0
"""Tetrachloromethane""","""ClC(Cl)(Cl)Cl""",-23.0,"""CCl4""",1,0,0,0,0,0,4,0,0,153.823,1,0,0,0,0,0,0,0,0,0
"""Iodine""","""II""",113.5,"""I2""",0,0,0,0,0,0,0,0,2,253.808,0,0,0,0,0,0,0,0,0,0


In [22]:
df.write_parquet("../data/processed/baseline_model_features.parquet")